# w9_mq_i2ce.ipynb — DEDICATED: the mq anchor-cap curve (8192 → 4096 → 2048)

User decree: the whole MoCo-supply cap curve lives here, most important cap
first. The bet: the noname axis scales LINEARLY with anchor cap (i2ce ZS non
0.642@512 → 0.667@2048 → 0.701@4096, no bend) while neutral saturates — and
only the mq supply path reaches g8192 (per-step anchor encoding is batch-only
192×cap no-grad through the shadow; the full-gallery-with-grad pass that
made g8192 cost ~72 GB is gone). Eval-time SGal at g8192 is ~34 GB fp16
on-GPU: use an **A100 80GB** pod.

Read two things off the curve: (1) does cap→noname linearity survive to
8192 (queue-internal comparison, the stale-key tax cancels); (2) does the
tax vs equal-cap re-encode shrink or hold (mq@2048 vs i2cce@2048 done rows).
Caps whose result json already exists are skipped (2048 is done: 0.799).
Claims are heartbeat-compatible with the campaign pods. AUTO-STOPS when the
curve is complete.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

ARM = "wcle_mq3072i2cce_icetf"
CAPS = [8192, 4096, 2048]              # user order: the important cap FIRST
EPOCHS = 2000                          # user decree: the mq family is a slow
# burner (shadow-EMA warm-up; @2048 probe peaked 0.819@ep900 STILL RISING and
# vsel picked ep650) -- the whole curve gets a 2000-ep budget. The @2048 run
# EXTENDS from its archived ep1000 checkpoint (worker ckpt-fallback resume).
os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", [f"w9_{ARM}_g{c}" for c in CAPS], f"@ {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed for this arm).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
# Needed twice: training views AND the on-pod anchor builds (cap > 2048).
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- worker will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run the curve SEQUENTIALLY (one GPU), most important cap first, to EPOCHS.
# Uses J.extend_fs for every cap: claims (heartbeat), removes stale best-jsons
# so the head re-picks over ALL checkpoints, reruns the worker; fresh caps
# train from scratch, finished-at-1000 caps resume from their newest ckpt.
from pathlib import Path

for CAP in CAPS:
    NM = f"w9_{ARM}_g{CAP}"
    if (Path(OUT_DIR) / f"tower_{NM}_fp_ep{EPOCHS}.npz").exists()             and (Path(OUT_DIR) / J.result_name(NM)).exists():
        print(f"[skip] {NM} already complete at {EPOCHS}ep"); continue
    J.extend_fs((ARM, CAP, False, 0, "clean", 16), EPOCHS,
                repo=REPO, data_dir=DATA_DIR, out_fs=OUT_DIR,
                full_pool_path=FULL_POOL_PATH)


In [ ]:
# Readout: the mq anchor-cap curve + equal-cap tax, per variant.
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = ([(f"ft4var_w9_{ARM}_g{c}_fp", f"mq @{c}") for c in sorted(CAPS)]
        + [("ft4var_w9_wcle_i2cce_icetf_g2048_fp", "i2cce @2048 (re-encode ref)"),
           ("ft4var_w9_wcle_i2ce_icetf_g4096_fp", "i2ce @4096 (re-encode ref)")])
for stem, lab in ROWS:
    p = Path(OUT_DIR) / f"{stem}_best.json"
    if not p.exists():
        print(f"{lab}: (missing)"); continue
    d = json.loads(p.read_text())
    runs = d["per_seed"]
    r = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
    print(f"{lab:28s} ep{d.get('best_ep')} "
          + " ".join(f"{v[:3]}:{r[v]:.3f}" for v in VORD)
          + f" m4:{np.mean(list(r.values())):.3f}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
